# CVD_Active goal: fixed-question contextual Fusion answers

Select a process occurrence, its Step and an analysis-center sensor separately.
Keep the existing sensor-free English questions. Current/reference traces and all
context sensors enter Fusion; a trained contextual head returns answer logits.
q99/GMM are explicit numerical comparators, never fallback answers. Historical
outputs below predate this goal until rerun. No real fault quality is established.


# STEP 3.1 - Frozen GEVIS inference

Current/reference waveforms and all context sensors enter the encoder and question-conditioned Fusion. The trained selected-center head returns BOOLEAN, CHOICE and ORDINAL. No numerical gate chooses these answers.


## Current execution contract

Use the configured multiscale encoder and frozen question-conditioned Fusion.
STEP1: reconstruction pretraining, TRAIN normalization, Validation checkpoint selection.
STEP2: question/numeric localization pretraining; CHOICE_SENSOR is supporting localization,
not a fourth trained STEP3 answer head. STEP3 trains BOOLEAN/CHOICE/ORDINAL readouts
on TRAIN synthetic pairs and selects on Validation. STEP3.1 loads and predicts only.
STEP4 evaluates the inspected Test benchmark; EXP1-4 diagnose behavior, EXP5 measures
PECVD requests, EXP6 transfers to PEALD, EXP7 scores external injections after prediction,
and EXP8 replays new inputs. Preserve compatible existing weights; retrain only when
intentionally changing a training contract. Restart kernels before Run All.

Questions remain fixed English strings; center/Step selection is numeric context.
Final answers are learned logits/argmax. Attention is supporting evidence, not fault truth.
Current Test is a development benchmark after earlier Test-guided changes, even with
disjoint runs. Fresh untouched runs are required for confirmatory claims.


In [ ]:
# Shared settings: edit config/experiments/gevis_pipeline.json, then restart kernels.
from pathlib import Path
import sys, os
PIPELINE_ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(PIPELINE_ROOT/'src') not in sys.path: sys.path.insert(0,str(PIPELINE_ROOT/'src'))
import importlib
for _name in ("pecvd.model_variants", "pecvd.experiments.questions", "pecvd.experiments.pipeline_settings"):
    importlib.reload(importlib.import_module(_name))
from pecvd.experiments.pipeline_settings import configure_pipeline
PIPELINE_SETTINGS=configure_pipeline(PIPELINE_ROOT)
print({k:PIPELINE_SETTINGS[k] for k in ('model_variant','training_profile','backbone_revision','decision_system') if k in PIPELINE_SETTINGS})

from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM, KINDS, QUESTIONS
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('Active notebooks require decision_system=fusion_answers.')
print({'question_contract':PRIMARY_GOAL_VERSION,'gevis_system':GEVIS_SYSTEM,'decision_path':'learned Fusion readout',
       'questions':{k:QUESTIONS[k] for k in KINDS},'answer_types':KINDS,
       'training':'STEP3 TRAIN readout; Validation selection; frozen STEP1/2',
       'interval_mode':'unavailable in this answer path',
       'probabilities':'softmax scores; not calibrated fault probabilities'})


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict
import copy, hashlib, json, math, random, sys, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir()),None)
if ROOT is None: raise RuntimeError('Run inside the PECVD project')
sys.path.insert(0,str(ROOT/'src'))
from pecvd.process_config import load_process_config
import importlib
import pecvd.answer_display as answer_text
answer_text=importlib.reload(answer_text)
import pecvd.gevis_fast as gevis_fast
gevis_fast=importlib.reload(gevis_fast)
from pecvd.gevis_fast import frozen_text
import pecvd.gevis_backbone as gevis_backbone
import importlib
gevis_backbone=importlib.reload(gevis_backbone)
render_semantic_question=gevis_backbone.render_semantic_question
from pecvd.experiments.process_inputs import configured_process_path,candidate_backbone_path
PROCESS_CONFIG_PATH=configured_process_path(ROOT)
import importlib
import pecvd.model_variants as model_variants
importlib.reload(model_variants)
from pecvd.model_variants import active_config,active_run_tag,selected_variant
PROCESS_CONFIG=active_config(load_process_config(PROCESS_CONFIG_PATH))
DATA_PATH=ROOT/PROCESS_CONFIG.data_path
ENCODER_ARTIFACT_VERSION=PROCESS_CONFIG.artifacts.encoder
ENCODER_ROOT=ROOT/'artifacts'/ENCODER_ARTIFACT_VERSION
RUN_COLUMN=PROCESS_CONFIG.columns.run; STEP_COLUMN=PROCESS_CONFIG.columns.step; ORDER_COLUMN=PROCESS_CONFIG.columns.order
STEPS=PROCESS_CONFIG.step_specs

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def save_torch(obj,path,overwrite=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(f'{path} exists; set OVERWRITE_ARTIFACTS=True intentionally')
    tmp=path.with_suffix(path.suffix+'.tmp'); torch.save(obj,tmp); tmp.replace(path)

def save_json(obj,path,overwrite=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(f'{path} exists; set OVERWRITE_ARTIFACTS=True intentionally')
    tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(path)

def derive_step_label(frame):
    if STEP_COLUMN in frame: return frame
    prefix=PROCESS_CONFIG.columns.step_one_hot_prefix; excluded=set(PROCESS_CONFIG.columns.step_one_hot_excluded)
    cols=[x for x in frame if x.startswith(prefix) and x not in excluded]
    if not cols: raise KeyError('step_label and step_name_* are both missing')
    values=frame[cols].fillna(0).to_numpy(); ix=values.argmax(1); strength=values.max(1)
    out=frame.copy(); labels=np.array([x.removeprefix(prefix) for x in cols],object)[ix]
    out[STEP_COLUMN]=np.where(strength>0,labels,None); return out

def select_sensor_columns(frame):
    forbidden={RUN_COLUMN,STEP_COLUMN,ORDER_COLUMN,'row_in_step','local_sample_id','global_sample_id','local_process_id','global_process_id','sample_order','sample_order_within_timestamp','timestamp_group_size','exact_value_group_size','is_exact_duplicate','step_elapsed_seconds','step_row_count','elapsed_seconds','elapsed_from_declared_start_seconds','elapsed_from_first_sample_seconds','data_split','step_segment_id','step_number','step_progress','previous_step_label','previous_step_number','previous_step_duration','previous_step_row_count','previous_step_end_order'}
    prefixes=('step_name_','source_group_','previous_step_','pm_cycle_','recipe_','label_','target_')
    cols=[x for x in frame.select_dtypes(include=np.number) if x not in forbidden and not x.startswith(prefixes)]
    if not cols: raise ValueError('No physical numeric sensor columns remain')
    return cols

def split_run_ids(frame,seed=SEED):
    ids=np.array(sorted(frame[RUN_COLUMN].dropna().unique().tolist()),dtype=object); np.random.default_rng(seed).shuffle(ids)
    n=len(ids); a=max(1,int(.70*n)); b=max(a+1,a+int(.15*n))
    result={'train':ids[:a].tolist(),'validation':ids[a:b].tolist(),'test':ids[b:].tolist()}
    sets={k:set(v) for k,v in result.items()}; assert not sets['train']&sets['validation'] and not sets['train']&sets['test'] and not sets['validation']&sets['test']
    return result

class PatchEncoder(nn.Module):
    def __init__(self,n_sensors,patch_len,d_model=64,nhead=4,layers=2,dropout=.1):
        super().__init__(); self.n_sensors=n_sensors; self.patch_len=patch_len; self.d_model=d_model
        self.input=nn.Linear(patch_len,d_model); self.sensor_emb=nn.Embedding(n_sensors,d_model); self.patch_emb=nn.Embedding(64,d_model); self.mask_token=nn.Parameter(torch.zeros(d_model))
        block=nn.TransformerEncoderLayer(d_model,nhead,4*d_model,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(block,layers); self.norm=nn.LayerNorm(d_model); self.decoder=nn.Linear(d_model,patch_len)
    def patchify(self,x):
        b,t,s=x.shape; p=t//self.patch_len
        return x[:,:p*self.patch_len].reshape(b,p,self.patch_len,s).permute(0,3,1,2).reshape(b,s*p,self.patch_len),p
    def tokens(self,x,mask=None):
        z,p=self.patchify(x); b=z.shape[0]; sid=torch.arange(self.n_sensors,device=x.device).repeat_interleave(p); pid=torch.arange(p,device=x.device).repeat(self.n_sensors)
        content=self.input(z)
        if mask is not None: content=torch.where(mask[...,None],self.mask_token[None,None],content)
        h=content+self.sensor_emb(sid)[None]+self.patch_emb(pid)[None]
        return self.norm(self.encoder(h))
    def forward(self,x,mask=None): return self.decoder(self.tokens(x,mask))

def deterministic_mask(batch,tokens,ratio,seed,device):
    g=torch.Generator(device=device); g.manual_seed(seed)
    mask=torch.rand((batch,tokens),generator=g,device=device)<ratio; empty=~mask.any(dim=1)
    if empty.any():
        rows=empty.nonzero(as_tuple=True)[0]; fallback=torch.randint(0,tokens,(len(rows),),generator=g,device=device); mask[rows,fallback]=True
    return mask

def masked_metrics(model,x,ratio=.25,seed=0):
    target,_=model.patchify(x); mask=deterministic_mask(len(x),target.shape[1],ratio,seed,x.device)
    pred=model(x,mask); den=mask.sum().clamp_min(1)*target.shape[-1]
    mse=((pred-target).pow(2)*mask[...,None]).sum()/den
    zero=(target.pow(2)*mask[...,None]).sum()/den
    return mse,zero,mask,pred,target

FDC_FEATURE_NAMES=('current_patch_mean','reference_patch_mean','absolute_mean_delta','current_patch_std','reference_patch_std','mean_absolute_point_delta','current_reconstruction_mse','reference_reconstruction_mse')

def numeric_state(encoder,current,reference):
    from pecvd.temporal_bridge import supports, numeric_state as temporal_state
    if supports(encoder): return temporal_state(encoder,current,reference)
    a=encoder.tokens(current); b=encoder.tokens(reference)
    current_patch,_=encoder.patchify(current); reference_patch,_=encoder.patchify(reference)
    current_reconstruction=encoder(current); reference_reconstruction=encoder(reference)
    fdc_features=torch.stack([current_patch.mean(-1),reference_patch.mean(-1),(current_patch.mean(-1)-reference_patch.mean(-1)).abs(),current_patch.std(-1,unbiased=False),reference_patch.std(-1,unbiased=False),(current_patch-reference_patch).abs().mean(-1),(current_reconstruction-current_patch).pow(2).mean(-1),(reference_reconstruction-reference_patch).pow(2).mean(-1)],-1)
    return torch.cat([a,b,a-b,(a-b).abs(),fdc_features],-1)

ANSWER_TYPES={'BOOLEAN':0,'CHOICE':1,'ORDINAL':2}
CRITERIA={k:list(v) for k,v in gevis_fast.DEFAULT_ANSWER_CRITERIA.items()}
class ByteTextEncoder(nn.Module):
    def __init__(self,d_model=128,max_bytes=320,layers=2,dropout=.1):
        super().__init__(); self.max_bytes=max_bytes; self.token=nn.Embedding(257,d_model,padding_idx=0); self.position=nn.Embedding(max_bytes,d_model)
        block=nn.TransformerEncoderLayer(d_model,4,4*d_model,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(block,layers); self.norm=nn.LayerNorm(d_model)
    def forward(self,texts,device):
        encoded=[list(text.encode('utf-8')[:self.max_bytes]) for text in texts]; width=max(1,max(map(len,encoded)))
        ids=torch.zeros((len(texts),width),dtype=torch.long,device=device)
        for i,raw in enumerate(encoded):
            if raw: ids[i,:len(raw)]=torch.tensor([x+1 for x in raw],device=device)
        mask=ids.eq(0); pos=torch.arange(width,device=device)[None]
        z=self.encoder(self.token(ids)+self.position(pos),src_key_padding_mask=mask)
        keep=(~mask).unsqueeze(-1); return self.norm((z*keep).sum(1)/keep.sum(1).clamp_min(1))

class GEVIS(nn.Module):
    def __init__(self,numeric_dim=256,d_model=128,dropout=.1,text_encoder_name='byte',text_encoder_revision=None):
        super().__init__(); self.text_encoder_name=text_encoder_name; self.text_encoder_revision=text_encoder_revision; self.text=ByteTextEncoder(d_model,dropout=dropout) if text_encoder_name=='byte' else gevis_backbone.E5TextEncoder(d_model,model_name=text_encoder_name,revision=text_encoder_revision or gevis_backbone.DEFAULT_TEXT_ENCODER_REVISION); self.numeric_input_norm=nn.LayerNorm(numeric_dim); self.numeric=nn.Linear(numeric_dim,d_model); self.numeric_key_norm=nn.LayerNorm(d_model); self.descriptor_norm=nn.LayerNorm(d_model); self.fusion_gate=nn.Sequential(nn.Linear(2*d_model,d_model),nn.Sigmoid()); self.type_emb=nn.Embedding(3,d_model)
        from pecvd.temporal_bridge import install
        install(self,numeric_dim,d_model)
        self.cross=nn.MultiheadAttention(d_model,4,dropout,batch_first=True); self.answer_norm=nn.LayerNorm(d_model); self.support_head=nn.Sequential(nn.LayerNorm(d_model),nn.Linear(d_model,64),nn.GELU(),nn.Linear(64,1)); self.change_head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,d_model),nn.GELU(),nn.Dropout(dropout),nn.Linear(d_model,1))
        self.semantic_heads=nn.ModuleDict({name:nn.Linear(d_model,size) for name,size in {'statistic':5,'direction':3,'time_scope':3}.items()})
        self.criterion_proj=nn.Sequential(nn.Linear(d_model,d_model),nn.GELU(),nn.LayerNorm(d_model)); self.temperature=nn.Parameter(torch.tensor(0.0))
    def load_state_dict(self,*args,**kwargs):
        self._text_cache=None
        return super().load_state_dict(*args,**kwargs)
    def _text(self,texts,device): return frozen_text(self,texts,device)
    def support_probability(self,questions): return self.support_head(self._text(questions,next(self.parameters()).device)).squeeze(-1).sigmoid()
    def forward(self,numeric_tokens,questions,answer_types,criteria,token_descriptions=None,return_fusion=False):
        device=numeric_tokens.device; type_ids=torch.tensor([ANSWER_TYPES[x] for x in answer_types],device=device)
        if numeric_tokens.shape[-1]!=self.numeric.in_features: raise ValueError(f'numeric state width {numeric_tokens.shape[-1]} does not match GEVIS input width {self.numeric.in_features}; rerun STEP 2 and STEP 3 with the current rich-SSL contract')
        text_query=self._text(questions,device); q=text_query+self.type_emb(type_ids); numeric_keys=self.numeric_key_norm(self.numeric(self.numeric_input_norm(numeric_tokens))); descriptors=None
        if token_descriptions is not None:
            if len(token_descriptions)!=len(numeric_tokens) or any(len(items)!=numeric_tokens.shape[1] for items in token_descriptions): raise ValueError('token_descriptions must match [batch, numeric_tokens]')
            flat_descriptions=[item for items in token_descriptions for item in items]
            unique=list(dict.fromkeys(flat_descriptions)); lookup={text:index for index,text in enumerate(unique)}
            indices=torch.tensor([lookup[text] for text in flat_descriptions],device=device,dtype=torch.long)
            descriptors=self.descriptor_norm(self._text(unique,device).index_select(0,indices).reshape_as(numeric_keys))
        from pecvd.gevis_backbone import attention_inputs
        attention_query,keys=attention_inputs(numeric_keys,descriptors,text_query,q,self.fusion_gate,getattr(self,'fusion_mode','legacy'))
        answer,attention=self.cross(attention_query[:,None],keys,keys,need_weights=True); answer=self.answer_norm(answer[:,0]+q)
        fusion=(answer,q,attention[:,0])
        if return_fusion: return fusion
        return self.decode(fusion,answer_types,criteria),attention[:,0]
    def decode(self,fusion,answer_types,criteria):
        answer,q,_=fusion; device=answer.device
        change_logit=self.change_head(torch.cat([answer,q],-1)).squeeze(-1)
        scale=self.temperature.exp().clamp(1e-2,100)
        flat=[item for items in criteria for item in items]; all_emb=self.criterion_proj(self._text(flat,device)); logits=[]; offset=0
        for i,items in enumerate(criteria):
            emb=all_emb[offset:offset+len(items)]; offset+=len(items); score=scale*(emb@answer[i])/math.sqrt(answer.shape[-1])
            if answer_types[i]=='BOOLEAN' and len(items)==2: score=score+torch.stack([-change_logit[i],change_logit[i]])
            logits.append(score)
        return logits

LJR_NAME='Learning from Joint Rewards'
@dataclass(frozen=True)
class GevisQuestion:
    text:str
    answer_type:str
    criteria:tuple[str,...]
    sensor:str|None=None
    statistic:str|None=None
    direction:str|None=None
    time_scope:str|None=None
    def validate(self):
        if not self.text.strip(): raise ValueError('Question text is empty')
        if self.answer_type not in ANSWER_TYPES: raise ValueError(f'answer_type must be {tuple(ANSWER_TYPES)}')
        if len(self.criteria)<2 or len(set(self.criteria))!=len(self.criteria): raise ValueError('criteria need at least two unique items')
        if self.statistic is not None and self.statistic not in {'level','variability','slope','local_change','residual'}: raise ValueError('Unsupported statistic')
        if self.direction is not None and self.direction not in {'increase','decrease','either'}: raise ValueError('Unsupported direction')
        if self.time_scope is not None and self.time_scope not in {'whole','early','late'}: raise ValueError('Unsupported time_scope')

@dataclass(frozen=True)
class ReferenceInfo:
    description:str
    process_count:int

## 1. 모델 및 identity 로드

In [ ]:
BACKBONE_ARTIFACT_VERSION='gevis_backbone_e5_v1_fdc_semantic_step1_bridge_all_types_v1_'+active_run_tag()
import os
import importlib
import pecvd.answer_display as answer_text
answer_text=importlib.reload(answer_text)
import pecvd.gevis_fast as gevis_fast
gevis_fast=importlib.reload(gevis_fast)
BACKBONE_PATH=Path(globals().get('EXPERIMENT_BACKBONE_PATH',candidate_backbone_path(ROOT,PROCESS_CONFIG,active_run_tag())))
MANIFEST_PATH=ENCODER_ROOT/'encoder_manifest.json'
CRITERIA={k:list(v) for k,v in gevis_fast.DEFAULT_ANSWER_CRITERIA.items()}

def load_runtime():
    if not BACKBONE_PATH.is_file(): raise FileNotFoundError(f'Missing STEP 2 v9 backbone: {BACKBONE_PATH}')
    if not MANIFEST_PATH.is_file(): raise FileNotFoundError(f'Missing STEP 1 manifest: {MANIFEST_PATH}')
    backbone_ck=torch.load(BACKBONE_PATH,map_location='cpu',weights_only=False)
    manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    if backbone_ck.get('schema')!='gevis.backbone.generic.v10': raise ValueError('Expected sensor-free generic-discovery STEP 2 backbone v10')
    if backbone_ck.get('step1_bridge',{}).get('manifest_sha256')!=sha256_file(MANIFEST_PATH): raise ValueError('STEP 1/2 manifest identity mismatch')
    if backbone_ck.get('text_encoder_name')!=gevis_backbone.DEFAULT_TEXT_ENCODER or backbone_ck.get('text_encoder_revision')!=gevis_backbone.DEFAULT_TEXT_ENCODER_REVISION: raise ValueError('E5 text encoder identity mismatch')
    model=GEVIS(int(backbone_ck['numeric_dim']),int(backbone_ck['d_model']),0.,backbone_ck['text_encoder_name'],backbone_ck['text_encoder_revision'])
    state=backbone_ck['backbone_state']
    import importlib
    import pecvd.temporal_bridge as checkpoint_bridge
    checkpoint_bridge=importlib.reload(checkpoint_bridge)
    checkpoint_bridge.install_checkpoint_adapter(model,state,int(backbone_ck['numeric_dim']),int(backbone_ck['d_model']))
    component=lambda name:{key.removeprefix(name+'.'):value for key,value in state.items() if key.startswith(name+'.')}
    for target,name in ((model.text,'text'),(model.numeric_input_norm,'numeric_input_norm'),(model.numeric,'numeric'),(model.numeric_key_norm,'numeric_key_norm'),(model.descriptor_norm,'descriptor_norm'),(model.fusion_gate,'fusion_gate'),(model.type_emb,'type_emb'),(model.cross,'cross'),(model.answer_norm,'norm')): target.load_state_dict(component(name))
    heads=backbone_ck['pretrain_heads_state']
    model.support_head.load_state_dict({key.removeprefix('support_head.'):value for key,value in heads.items() if key.startswith('support_head.')})
    model.semantic_heads.load_state_dict({key.removeprefix('semantic_heads.'):value for key,value in heads.items() if key.startswith('semantic_heads.')})
    model.fusion_mode=backbone_ck.get('fusion_mode','legacy')
    model.requires_grad_(False).eval(); model._text_cache=None
    encoders={}
    if manifest.get('schema')!='gevis.encoder_manifest.rich_ssl.v1' or set(manifest['step_checkpoints'])!=set(STEPS): raise ValueError('STEP 1 manifest/configuration mismatch')
    for step,item in manifest['step_checkpoints'].items():
        path=ENCODER_ROOT/item['path']
        if sha256_file(path)!=item['sha256']: raise ValueError(f'Encoder identity mismatch: {step}')
        eck=torch.load(path,map_location='cpu',weights_only=False); cfg=eck['model_config']
        from pecvd.encoder_loading import load_encoder  # Refresh local factories before strict offline reconstruction.
        encoder=load_encoder(eck,PatchEncoder); encoder.requires_grad_(False).eval(); encoders[step]=(encoder,eck)
    backbone_ck['_artifact_sha256']=sha256_file(BACKBONE_PATH)
    return model,encoders,backbone_ck

GEVIS_MODEL,STEP_ENCODERS,GEVIS_CHECKPOINT=load_runtime()
# Active workspace goal is configured explicitly; a missing head never enables rules.
from pecvd.experiments.pipeline_settings import configure_pipeline
DECISION_SYSTEM=configure_pipeline(ROOT).get('decision_system','hybrid')
if DECISION_SYSTEM=='fusion_answers':
    from pecvd.experiments.fusion_answers import attach, ANSWER_CRITERIA as FUSION_CRITERIA
    import sys
    attach(sys.modules[__name__])
    CRITERIA={k:list(v) for k,v in FUSION_CRITERIA.items()}
    DECISION_POLICY_MODE='fusion_answers'
    REFERENCE_POLICY=None
    DISCOVERY_POLICY=None
    DISCOVERY_POLICY_MODE='fusion_answers'
    DISCOVERY_POLICY_PATH=None
    INTERVAL_MODE='unavailable'
    TEMPORAL_LOCALIZER=None
    GEVIS_ANSWER_AVAILABLE=FUSION_ANSWER_HEAD is not None
    print({'decision_system':DECISION_SYSTEM,'trained_answer_head_available':GEVIS_ANSWER_AVAILABLE,
           'answer_checkpoint':str(FUSION_ANSWER_PATH),'rule_fallback':False})
else:
    # Restore trained decoder into the SAME model; numeric discovery remains a separate path.
    GEVIS_ANSWER_PATH=BACKBONE_PATH.parent/'gevis_answer_model.pt'
    GEVIS_ANSWER_AVAILABLE=False
    if GEVIS_ANSWER_PATH.is_file() and os.environ.get('GEVIS_ENABLE_ANSWER_ABLATION')=='1':
        from pecvd.gevis_fast import load_answer_checkpoint
        load_answer_checkpoint(GEVIS_MODEL,GEVIS_ANSWER_PATH,backbone_sha256=GEVIS_CHECKPOINT['_artifact_sha256'])
        GEVIS_ANSWER_AVAILABLE=True
    print({'trained_answer_decoder_available':GEVIS_ANSWER_AVAILABLE,'answer_checkpoint':str(GEVIS_ANSWER_PATH),'numerical_discovery':'separate calibrated-evidence path'})
    print({'schema':GEVIS_CHECKPOINT['schema'],'runtime':'predicted semantic slots + predicted sensor + explicit operation','steps':sorted(STEP_ENCODERS),'answer_types':tuple(CRITERIA)})

    import os
    DECISION_POLICY_MODE=os.environ.get('GEVIS_DECISION_POLICY','reference_quantiles_v2')
    if DECISION_POLICY_MODE not in ('legacy','reference_quantiles_v1','reference_quantiles_v2'): raise ValueError('Unknown decision policy')
    if DECISION_POLICY_MODE in ('reference_quantiles_v1','reference_quantiles_v2'):
        if DECISION_POLICY_MODE=='reference_quantiles_v2':
            from pecvd.experiments.reference_policy_v2 import ReferenceConfig,identity_for,load_policy
        else:
            from pecvd.experiments.reference_policy import ReferenceConfig,identity_for,load_policy
        REFERENCE_POLICY_PATH=Path(os.environ.get('GEVIS_REFERENCE_POLICY_PATH',str(BACKBONE_PATH.parent/('reference_policy_v2.json' if DECISION_POLICY_MODE=='reference_quantiles_v2' else 'reference_policy_v1.json'))))
        if not REFERENCE_POLICY_PATH.is_file(): raise FileNotFoundError('Reference criteria missing. Run STEP 3 reference-policy preparation; STEP 1/2 retraining is not needed.')
        REFERENCE_CONFIG=ReferenceConfig(**json.loads(REFERENCE_POLICY_PATH.read_text(encoding='utf-8'))['config'])
        REFERENCE_POLICY=load_policy(REFERENCE_POLICY_PATH,identity_for(ROOT,ENCODER_ROOT,BACKBONE_PATH,REFERENCE_CONFIG))

    # Separate automatic-window criteria; named-question reference policy is unchanged.
    from types import SimpleNamespace as _DiscoveryNamespace
    from pecvd.experiments.discovery_calibration import attach_policy as _attach_discovery_policy
    _discovery_state=_DiscoveryNamespace(BACKBONE_PATH=BACKBONE_PATH,ENCODER_ROOT=ENCODER_ROOT)
    _attach_discovery_policy(_discovery_state)
    DISCOVERY_POLICY_MODE=_discovery_state.DISCOVERY_POLICY_MODE
    DISCOVERY_POLICY=_discovery_state.DISCOVERY_POLICY
    DISCOVERY_POLICY_PATH=getattr(_discovery_state,'DISCOVERY_POLICY_PATH',None)
    if DISCOVERY_POLICY_MODE=='window_calibrated_v2' and DISCOVERY_POLICY is None:
        print('Automatic window policy not prepared: run STEP 3 window-evidence-build; no fallback to marginal alarms.')

    # Runtime loads a trained interval module only; never fits on inference input.
    from pecvd.experiments.temporal_localization import attach_localizer
    _localization_state=_DiscoveryNamespace(BACKBONE_PATH=BACKBONE_PATH,ENCODER_ROOT=ENCODER_ROOT)
    attach_localizer(_localization_state)
    INTERVAL_MODE=_localization_state.INTERVAL_MODE
    TEMPORAL_LOCALIZER=_localization_state.TEMPORAL_LOCALIZER
    TEMPORAL_LOCALIZER_PATH=getattr(_localization_state,'TEMPORAL_LOCALIZER_PATH',None)


## Learned answers for an explicitly selected analysis center

Select process/run, Step and sensor outside the question text. Use ask_gevis with
center_sensor for one exact model window, or review_selected_step for the complete
selected Step timeline. Fixed question text and trained option order are required.


In [ ]:
STATISTIC_NAMES=('level','variability','slope','local_change','residual')
DIRECTION_NAMES=('increase','decrease','either')
SCOPE_NAMES=('whole','early','late')

def prepare_window(rows,meta):
    missing=[x for x in meta['sensor_columns'] if x not in rows]
    if missing: raise KeyError(f'{len(missing)} required sensors are missing')
    ordered=rows.sort_values(ORDER_COLUMN).tail(meta['window_length']); x=ordered[meta['sensor_columns']].to_numpy(np.float32)
    if len(x)!=meta['window_length']: raise ValueError(f"Need at least {meta['window_length']} rows")
    if not np.isfinite(x).all(): raise ValueError('Window contains NaN or infinite sensor values')
    if 'row_in_step' in ordered and 'step_row_count' in ordered:
        start=float(ordered['row_in_step'].iloc[0]); total=float(ordered['step_row_count'].iloc[0]); ratio=start/max(1,total-meta['window_length'])
    else: ratio=1.0
    position_bin=min(9,max(0,int(ratio*10))); normalized=((x-meta['mean'])/meta['std']).astype(np.float32)
    return torch.from_numpy(normalized)[None],{'position_bin':position_bin,'position_ratio':float(ratio)}

def aggregate_reference_windows(reference_rows,meta):
    if not reference_rows: raise ValueError('At least one reference Process is required')
    ordered=[rows.sort_values(ORDER_COLUMN).tail(meta['window_length']).copy() for rows in reference_rows]
    if any(len(rows)!=meta['window_length'] for rows in ordered): raise ValueError('Every reference needs one complete aligned window')
    arrays=[rows[meta['sensor_columns']].to_numpy(np.float32) for rows in ordered]
    if not all(np.isfinite(values).all() for values in arrays): raise ValueError('Reference window contains NaN or infinite sensor values')
    result=ordered[0].copy(); result.loc[:,meta['sensor_columns']]=np.median(np.stack(arrays),axis=0); return result

def numeric_token_descriptions(step,meta,patch_count,batch_size):
    names=meta.get('sensor_descriptors',{name:name for name in meta['sensor_columns']}); phase=meta.get('step_descriptor',step)
    one=gevis_backbone.token_descriptions(phase,[names.get(sensor,sensor) for sensor in meta['sensor_columns']],patch_count)
    return [one for _ in range(batch_size)]

def semantic_comparison_features(current,reference,state,patch_count):
    from pecvd.experiments.operations import semantic_comparison_features as shared_operation
    if globals().get('DECISION_POLICY_MODE','legacy') in ('reference_quantiles_v1','reference_quantiles_v2'):
        from pecvd.experiments.reference_policy import policy_features
        return policy_features(current,reference,state,patch_count)
    return shared_operation(current,reference,state,patch_count)

def make_grounded_questions(step,sensor=None,*,expanded=False,limit=120):
    from pecvd.experiments.questions import discovery_questions
    if step not in STEP_ENCODERS: raise KeyError(step)
    return discovery_questions(GevisQuestion)

def make_discovery_questions():
    # Default automatic review: no target sensor is supplied in question text.
    from pecvd.experiments.questions import discovery_questions
    return discovery_questions(GevisQuestion)

@torch.no_grad()
def ask_gevis(current_rows,reference_rows,step,questions,reference_info,confidence_floor=.55,semantic_floor=.60,grounding_floor=.30,support_floor=.50,allow_incomparable=False,*,routing_mode="learned",center_sensor=None):
    if globals().get('DECISION_SYSTEM','hybrid')=='fusion_answers':
        import sys
        from pecvd.experiments.fusion_answers import answer_questions
        return answer_questions(sys.modules[__name__],current_rows,reference_rows,step,questions,
            reference_info,center_sensor=center_sensor)

    from pecvd.experiments.routing_audit import ROUTING_MODES,explicit_slots,acceptance_gates
    if routing_mode not in ROUTING_MODES: raise ValueError('Unknown routing mode')
    inference_started=time.perf_counter()
    if step not in STEP_ENCODERS: raise KeyError(step)
    if not isinstance(reference_info,ReferenceInfo): raise TypeError('reference_info is required')
    if reference_info.process_count<1: raise ValueError('reference process_count must be positive')
    model=GEVIS_MODEL.to(DEVICE).eval(); encoder,meta=STEP_ENCODERS[step]; encoder=encoder.to(DEVICE).eval()
    from pecvd.experiments.evidence import prepare_inputs,pointwise_evidence
    current,reference,current_context,reference_context,state,patch_count,features=prepare_inputs(
        current_rows,reference_rows,step,encoder,meta,DEVICE,prepare_window,numeric_state,semantic_comparison_features)
    comparable=abs(current_context['position_bin']-reference_context['position_bin'])<=1
    if not comparable and not allow_incomparable: raise ValueError('Incomparable reference position')
    for question in questions: question.validate()
    texts=[question.text for question in questions]; shared_state=state.expand(len(questions),-1,-1); descriptions=numeric_token_descriptions(step,meta,patch_count,len(questions))
    if not questions: raise ValueError("At least one question is required")
    fused_batch=model(shared_state,texts,[q.answer_type for q in questions],[list(q.criteria) for q in questions],descriptions,return_fusion=True)
    fusions=[tuple(part[i:i+1] for part in fused_batch) for i in range(len(questions))]
    text_embedding=model._text(texts,DEVICE); answers=[]
    names=meta.get('sensor_descriptors',{name:name for name in meta['sensor_columns']})
    for index,(question,fusion) in enumerate(zip(questions,fusions)):
        attention=fusion[2][0].reshape(len(meta['sensor_columns']),patch_count); sensor_scores=attention.sum(-1); sensor_probability=sensor_scores/sensor_scores.sum().clamp_min(1e-8); sensor_id=int(sensor_probability.argmax()); patch_id=int(attention[sensor_id].argmax())
        semantic_prob={name:model.semantic_heads[name](text_embedding[index]).softmax(-1) for name in ('statistic','direction','time_scope')}
        statistic_id=int(semantic_prob['statistic'].argmax()); direction_id=int(semantic_prob['direction'].argmax()); scope_id=int(semantic_prob['time_scope'].argmax())
        learned_query={'sensor':meta['sensor_columns'][sensor_id],'time_scope':SCOPE_NAMES[scope_id]}
        resolved=explicit_slots(question.text,meta['sensor_columns'],names) if routing_mode=='explicit_text' else {}
        explicit_sensor=resolved.get('sensor') is not None
        explicit_scope=resolved.get('time_scope') is not None
        if explicit_sensor: sensor_id=meta['sensor_columns'].index(resolved['sensor'])
        if explicit_scope: scope_id=SCOPE_NAMES.index(resolved['time_scope'])
        patch_id=int(attention[sensor_id].argmax())
        selected=features[:,scope_id,sensor_id]; statistic=torch.tensor([statistic_id],device=DEVICE); direction=torch.tensor([direction_id],device=DEVICE)
        answer_probability,_=gevis_fast.structured_answer_probabilities(selected,statistic,direction,temperature=.75); probability=answer_probability[question.answer_type][0]; answer_id=int(probability.argmax())
        criterion=None; criterion_ready=True
        if globals().get('DECISION_POLICY_MODE','legacy') in ('reference_quantiles_v1','reference_quantiles_v2'):
            from pecvd.experiments.reference_policy import context_from_rows
            criterion=REFERENCE_POLICY.decide(selected[0].cpu().numpy(),step,meta['sensor_columns'][sensor_id],scope_id,statistic_id,DIRECTION_NAMES[direction_id],context_from_rows(current_rows))
            criterion_ready=criterion['supported'][question.answer_type]
            values=criterion['probabilities'][question.answer_type]
            probability=torch.tensor(values if values is not None else [1/len(question.criteria)]*len(question.criteria),device=DEVICE)
            answer_id=int(probability.argmax())
        answer_confidence=float(probability.max()); required_semantics=('time_scope',) if globals().get('DECISION_POLICY_MODE')=='reference_quantiles_v2' and question.answer_type!='BOOLEAN' else ('statistic','direction','time_scope'); semantic_confidence=min(float(semantic_prob[key].max()) for key in required_semantics); grounding_confidence=float(sensor_probability.max()); support_confidence=float(model.support_head(text_embedding[index]).sigmoid())
        criteria_match=tuple(question.criteria)==tuple(CRITERIA[question.answer_type])
        supported=criteria_match and support_confidence>=support_floor
        score_gate=globals().get('DECISION_POLICY_MODE','legacy')!='reference_quantiles_v2'
        gate_checks=acceptance_gates(criterion_ready=criterion_ready,criteria_match=criteria_match,
            support_confidence=support_confidence,comparable=comparable,answer_confidence=answer_confidence,
            semantic_scores={key:float(values.max()) for key,values in semantic_prob.items()},
            sensor_confidence=grounding_confidence,required_semantics=required_semantics,
            confidence_floor=confidence_floor,semantic_floor=semantic_floor,grounding_floor=grounding_floor,
            support_floor=support_floor,score_gate=score_gate,explicit_sensor=explicit_sensor,explicit_scope=explicit_scope)
        accepted=all(check['passed'] for check in gate_checks.values())
        warnings=[]
        if not criterion_ready: warnings.append('reference_criterion_unsupported')
        if not supported: warnings.append('unsupported_question_or_criteria')
        if score_gate and answer_confidence<confidence_floor: warnings.append('low_answer_confidence')
        if any(not gate_checks['semantic_'+key]['passed'] for key in required_semantics): warnings.append('low_semantic_confidence')
        if not gate_checks['sensor_binding']['passed']: warnings.append('low_sensor_grounding_confidence')
        if not comparable: warnings.append('incomparable_reference_override')
        scope_start,scope_end=(0,meta['window_length']) if scope_id==0 else ((0,max(2,meta['window_length']//3)) if scope_id==1 else (max(0,2*meta['window_length']//3),meta['window_length']))
        declared={'sensor':question.sensor,'statistic':question.statistic,'direction':question.direction,'time_scope':question.time_scope}
        predicted={'sensor':meta['sensor_columns'][sensor_id],'sensor_description':names.get(meta['sensor_columns'][sensor_id],meta['sensor_columns'][sensor_id]),'statistic':STATISTIC_NAMES[statistic_id],'direction':DIRECTION_NAMES[direction_id],'time_scope':SCOPE_NAMES[scope_id]}
        if criterion is not None:
            from pecvd.experiments.reference_policy import policy_pointwise_evidence
            observed=policy_pointwise_evidence(current,reference,sensor_id,scope_id,criterion,original_scale=float(meta['std'][sensor_id]))
        else:
            observed=pointwise_evidence(current,reference,sensor_id,scope_id,original_scale=float(meta['std'][sensor_id]))
        answers.append({'question':question.text,'answer_type':question.answer_type,'answer':question.criteria[answer_id] if accepted else None,'candidate_answer':question.criteria[answer_id] if criterion_ready else None,'accepted':accepted,'probabilities':dict(zip(question.criteria,map(float,probability.cpu()))),'confidence':{'answer':answer_confidence,'semantic':semantic_confidence,'sensor_grounding':grounding_confidence,'question_support':support_confidence},'predicted_query':predicted,'semantic_scores':{name:dict(zip((STATISTIC_NAMES if name=='statistic' else DIRECTION_NAMES if name=='direction' else SCOPE_NAMES),map(float,values.detach().cpu()))) for name,values in semantic_prob.items()},'sensor_weights':dict(zip(meta['sensor_columns'],map(float,sensor_probability.detach().cpu()))),'reference_criterion':criterion,'score_meaning':('reference-relative operation scores; unsupported scores are uniform placeholders; not fault probabilities' if criterion is not None else 'LEGACY operation scores; not calibrated fault probabilities'),'declared_metadata_audit_only':declared,'operation':{'selected_feature_values':dict(zip(STATISTIC_NAMES,map(float,selected[0].cpu()))),'requested_value':float(selected[0,statistic_id]),'threshold':criterion['threshold'] if criterion is not None else 1.0,'meaning':'reference quantile comparison' if criterion is not None else 'LEGACY fixed-rule comparison'},'evidence':{'sensor':predicted['sensor'],'scope_rows':[scope_start,scope_end],'strongest_patch_index':patch_id,'pointwise':observed,'basis':'predicted sensor/scope + original FDC statistics + PatchTST residual','status':'accepted' if accepted else 'candidate_only'},'coverage_warnings':warnings,'gate_checks':gate_checks,'routing':{'mode':routing_mode,'learned_query':learned_query,'explicit':resolved,'sensor_source':'explicit_text' if explicit_sensor else 'cross_attention','scope_source':'explicit_text' if explicit_scope else 'semantic_head'},'step':step,'comparison_context':{'reference_scope':'position-matched median reference','position_bin_delta':abs(current_context['position_bin']-reference_context['position_bin'])},'reference':asdict(reference_info),'runtime_contract':('unique explicit sensor/scope text with learned fallback; no target metadata' if routing_mode=='explicit_text' else 'predicted semantic slots and predicted Cross-Attention sensor'),'checkpoint_sha256':GEVIS_CHECKPOINT['_artifact_sha256']})
    if DEVICE.type=='cuda': torch.cuda.synchronize(DEVICE)
    elapsed_ms=(time.perf_counter()-inference_started)*1000
    for answer in answers: answer['batch_inference_ms']=elapsed_ms
    return answers

def ask_gevis_against_pool(current_rows,reference_rows,step,question,confidence_floor=.55,*,center_sensor=None):
    if not isinstance(question,GevisQuestion): raise TypeError('Select exactly one GevisQuestion')
    meta=STEP_ENCODERS[step][1]; reference=aggregate_reference_windows(reference_rows,meta); info=ReferenceInfo('position-matched median reference',len(reference_rows))
    return ask_gevis(current_rows,reference,step,[question],info,confidence_floor,center_sensor=center_sensor)[0]


def review_changes(current_rows,reference_rows,step,reference_info,*,sensors=None,scopes=None,center_sensor=None):
    if globals().get('DECISION_SYSTEM','hybrid')=='fusion_answers':
        import sys
        from pecvd.experiments.fusion_answers import review
        if scopes is not None and list(scopes)!=['whole']:raise ValueError('Select the input window explicitly')
        return review(sys.modules[__name__],current_rows,reference_rows,step,reference_info,center_sensor=center_sensor)

    """UI sets the search range; evidence discovers sensor/type/location."""
    from pecvd.experiments.discovery import review_changes as shared_review
    from types import SimpleNamespace
    module=SimpleNamespace(**{name:globals()[name] for name in ('DECISION_POLICY_MODE','REFERENCE_POLICY','STEP_ENCODERS','DEVICE','prepare_window','numeric_state','GEVIS_MODEL','GEVIS_CHECKPOINT','numeric_token_descriptions')})
    module.DISCOVERY_POLICY_MODE=globals().get('DISCOVERY_POLICY_MODE','legacy_marginal_v1')
    module.DISCOVERY_POLICY=globals().get('DISCOVERY_POLICY')
    module.INTERVAL_MODE=globals().get('INTERVAL_MODE','numeric')
    module.TEMPORAL_LOCALIZER=globals().get('TEMPORAL_LOCALIZER')
    return shared_review(module,current_rows,reference_rows,step,reference_info,
        sensors=sensors,scopes=scopes,center_sensor=center_sensor)


def review_selected_step(frame,reference_step_rows,process_key,step,center_sensor,*,step_number=None):
    """Full selected process/Step timeline; explicit center, unchanged questions."""
    import sys
    from pecvd.experiments.fusion_answers import review_selected_step as learned_step
    return learned_step(sys.modules[__name__],frame,reference_step_rows,process_key,step,
        center_sensor,step_number=step_number)


## 3. Active runtime contract

Use the fixed English questions and an explicitly selected center. The model returns predictions without truth labels; independent truth is needed to score accuracy.


In [ ]:
from pecvd.experiments.fusion_answers import SYSTEM, KINDS, ANSWER_CRITERIA
print({'gevis_system':SYSTEM,'answer_types':KINDS,'criteria':ANSWER_CRITERIA,
       'interval_mode':INTERVAL_MODE,'trained_answer_head_available':GEVIS_ANSWER_AVAILABLE,
       'decision':'learned logits/argmax','rule_fallback':False})
# Exact aligned-window diagnostic, with an explicitly supplied reference:
# result=review_selected_step(frame,reference_step_rows,selected_process_key,selected_step,selected_sensor)
# For incoming complete model windows, use online_inference.InferenceSession as in EXP5-8.
# Its reference is selected from frozen TRAIN; no Test original or label is accepted.


## Interpretation

Yes means predicted change relative to the provided reference. It is not a confirmed equipment fault. Softmax scores are not calibrated correctness probabilities. ORDINAL mixes synthetic amplitude stages with native lag-sample stages; report them by type. Interval localization is unavailable in this path.


## Experiments

See `test/README.md`. EXP1-4 diagnose the fixed task; EXP5-8 measure or replay frozen model requests. Legacy numerical helper APIs below are optional comparators and never the active learned answer path.


In [ ]:
def inspect_observed_window(current_rows,reference_rows,step):
    from pecvd.experiments.evidence import prepare_inputs,evidence_survey
    encoder,meta=STEP_ENCODERS[step]; encoder=encoder.to(DEVICE).eval()
    current,reference,*_=prepare_inputs(current_rows,reference_rows,step,encoder,meta,DEVICE,prepare_window,numeric_state,semantic_comparison_features)
    return [{'sensor':meta['sensor_columns'][row['sensor_index']],**row} for row in evidence_survey(current,reference)]


In [ ]:
def review_boundary_comparison(current_rows,reference_rows,step,reference_info,**kwargs):
    """Compare q99 and direct GMM; restore q99; never fit on inference inputs."""
    import sys
    from pecvd.experiments.gmm_comparison import compare_pair
    return compare_pair(sys.modules[__name__],current_rows,reference_rows,step,reference_info,**kwargs)
